# Portable Colab build workspace

The host synchronizes the project through the native Colab CLI before this notebook runs. Build products remain in a persistent workspace for the lifetime of the Colab VM.

The host sends exactly one project Make target. This notebook does not interpret its name; it executes that target and optionally collects output when the host explicitly requests artifacts.

## Session setup

Load the request sent by the host and define the shared Make runner.

In [ ]:
from pathlib import Path, PurePosixPath
import base64
import json
import shutil
import subprocess
import sys

ROOT = Path('/content/.cloud-build/workspace')
SOURCE = ROOT / 'src'
CONTROL = Path('/content/cloud-build-target')
RESULT = ROOT.parent / 'target-result.json'
RESULT.unlink(missing_ok=True)

control_lines = CONTROL.read_text().splitlines()
if len(control_lines) not in (2, 4, 5, 7, 8):
    raise ValueError('invalid cloudmake control file')
TARGET_B64, JOBS = control_lines[:2]
REQUESTED_TARGET = base64.urlsafe_b64decode(TARGET_B64).decode()
MAKEFILE = control_lines[2] if len(control_lines) >= 4 else 'Makefile.build'
arguments_b64 = control_lines[3] if len(control_lines) >= 4 else 'W10='
collect_b64 = control_lines[4] if len(control_lines) in (5, 7, 8) else ''
image_b64 = control_lines[5] if len(control_lines) in (7, 8) else ''
devices_b64 = control_lines[6] if len(control_lines) in (7, 8) else 'W10='
runtimes_b64 = control_lines[7] if len(control_lines) == 8 else 'W10='
OCI_IMAGE = base64.urlsafe_b64decode(image_b64).decode() if image_b64 else None
OCI_DEVICES = json.loads(base64.urlsafe_b64decode(devices_b64).decode())
OCI_RUNTIMES = json.loads(base64.urlsafe_b64decode(runtimes_b64).decode())
COLLECT_DIR = base64.urlsafe_b64decode(collect_b64).decode() if collect_b64 else None
PROJECT_ARGUMENTS = json.loads(base64.urlsafe_b64decode(arguments_b64).decode())
if not REQUESTED_TARGET or '\n' in REQUESTED_TARGET or not isinstance(PROJECT_ARGUMENTS, list) or not all(isinstance(value, str) for value in PROJECT_ARGUMENTS):
    raise ValueError('invalid cloudmake target or project arguments')
if COLLECT_DIR is not None:
    collect_relative = PurePosixPath(COLLECT_DIR)
    if not COLLECT_DIR or collect_relative.is_absolute() or '..' in collect_relative.parts:
        raise ValueError('invalid cloudmake collection directory')

required_commands = ('tar',) if OCI_IMAGE else ('make', 'tar')
missing_commands = [name for name in required_commands if shutil.which(name) is None]
if missing_commands:
    raise RuntimeError(f'missing required remote command(s): {missing_commands}')


print(f'Workspace: {SOURCE}')
print(f'Requested target: {REQUESTED_TARGET}')
print(f'Parallel jobs: {JOBS}')

def run_make(target):
    if OCI_IMAGE:
        command = [
            sys.executable, '/content/cloudmake-oci-runner.py',
            '--mode', 'run', '--image', OCI_IMAGE, '--runtime', 'auto',
            '--source', str(SOURCE), '--cache', '/content/.cloud-build/oci-cache',
            '--makefile', MAKEFILE, '--rootless-workspace-owner', '65534:65534',
            '--cdi-spec-dir', '/content/.cloud-build/oci-cache/cdi',
            '--target-b64', TARGET_B64,
            '--arguments-b64', arguments_b64, '--jobs', JOBS,
            '--result', str(RESULT),
        ]
        for device in OCI_DEVICES:
            command.extend(['--device', device])
        for runtime in OCI_RUNTIMES:
            command.extend(['--runtime-candidate', runtime])
        print('+', ' '.join(command))
        completed = subprocess.run(
            command, check=False, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        print(completed.stdout, end='')
        return completed.returncode
    command = [
        'make', '-C', str(SOURCE), '-f', MAKEFILE,
        *PROJECT_ARGUMENTS,
        f'-j{JOBS}', '--', target,
    ]
    print('+', ' '.join(command))
    completed = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(completed.stdout, end='')
    return completed.returncode

## Project target

Pass the requested name to the project Makefile without a cloudmake target whitelist.

In [ ]:
artifact = Path('/content/.cloud-build/artifacts.tar.gz')
if COLLECT_DIR is not None:
    artifact.unlink(missing_ok=True)
TARGET_EXIT_CODE = run_make(REQUESTED_TARGET)
if not OCI_IMAGE:
    RESULT.parent.mkdir(parents=True, exist_ok=True)
    result_temporary = RESULT.with_suffix('.json.tmp')
    result_temporary.write_text(json.dumps({'schema': 1, 'target': REQUESTED_TARGET, 'exit_code': TARGET_EXIT_CODE}) + '\n')
    result_temporary.replace(RESULT)

## Optional artifact collection

Only the host `--collect DIR TARGET` operation requests an archive for download. The target name itself receives no special treatment.

In [ ]:
if TARGET_EXIT_CODE == 0 and COLLECT_DIR is not None:
    source_root = SOURCE.resolve()
    collect_path = (SOURCE / Path(*collect_relative.parts)).resolve()
    if collect_path != source_root and source_root not in collect_path.parents:
        raise ValueError('collection directory escapes the project')
    if not collect_path.is_dir():
        raise FileNotFoundError(f'collection directory does not exist: {COLLECT_DIR}')
    shutil.make_archive(str(artifact)[:-7], 'gztar', root_dir=collect_path)
    print(f'Artifacts ready at {artifact}')